# Managed Agents: the agentic loop and sessions

> **CCA-F Domain 1 - Agentic (27%).** The exam tests that you stop the loop on **`session.status_idle`** and read **`stop_reason`**, not on vibes. It also tests that a **session** carries memory server-side across turns, so you do not resend prior turns.

Theme: a **retro platformer video-game curator**. We stand up a **Managed Agent**, run one turn to idle, then run a second turn on the **same session** to prove the server remembers the first recommendation without us resending it.

### What Anthropic means by the "agentic loop"

Anthropic frames an agent as a model running in a loop over three repeating phases:

1. **Gather context** - the model reads the current state (your message, prior turns, tool results) to decide what to do next.
2. **Take action** - the model responds, or calls a tool, or hands off to a subagent.
3. **Verify work** - the model checks the outcome (a tool result, an error, a validation) and feeds it back into the next pass of the loop.

The loop keeps turning until the model has nothing left to do, at which point it **ends the turn**. What makes an agent an *agent* is that the model, not your code, decides when to loop again and when to stop.

**Where Managed Agents fit.** With the hand-rolled Messages API you run this loop yourself: you inspect `stop_reason`, execute tools, append results, and call the API again. With **Managed Agents** the server runs the loop for you. Each phase surfaces as a streamed **event**:

| Loop phase | What you send / see |
|---|---|
| Gather context | you `send` a `user.message` event; the **session** already holds prior turns |
| Take action | the stream yields `agent.message` (text) and, mid-turn, `agent.tool_use` events |
| Verify work | the runtime resolves tool results, then loops or ends |
| Loop ends | `session.status_idle` fires, carrying the authoritative **`stop_reason`** |

So the one rule that anchors this notebook: **the turn is over when `session.status_idle` fires**, not when the assistant text looks finished. That event is the loop's terminal signal, and its `stop_reason` tells you *why* it stopped (`end_turn`, `requires_action`, or `retries_exhausted`).

### 1. Install dependencies

In [1]:
# uv-managed venvs ship WITHOUT pip, so `%pip install` fails here.
# Use uv instead, pointed at THIS kernel's interpreter so packages land
# where the kernel actually looks. Idempotent: uv no-ops if already satisfied.
import subprocess, sys

subprocess.run(
    ["uv", "pip", "install", "--python", sys.executable, "anthropic", "python-dotenv"],
    check=True,
)
print("Dependencies ready.")

Dependencies ready.


### 2. Load environment variables from .env file

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

### 3. Create an API client

The Managed Agents surface is **beta-gated**. We set `BETA` to the beta header value once and pass `betas=[BETA]` on every call. `MODEL` stays on the repo default **Haiku 4.5**.

In [3]:
import os
from anthropic import Anthropic

client = Anthropic()
MODEL = "claude-haiku-4-5"          # repo default; Sonnet only where reasoning depth earns it
BETA = "managed-agents-2026-04-01"  # beta header for the whole Managed Agents surface

# Teardown guard. Default OFF so "Run All" in class leaves the agent and session
# live for Console inspection. Headless smoke tests set ARCHIVE_ON_RUN=1 to force
# cleanup so no billable resource dangles in CI. See the final cell.
RUN_TEARDOWN = os.environ.get("ARCHIVE_ON_RUN", "0") == "1"

### 4. Create the agent, the environment, and the session

Three server-side resources, three verbs. **`agents.create`** takes `model={"id": MODEL}` plus a `name`, with `system` optional. **`environments.create`** is the scratch space the runtime executes in. **`sessions.create`** binds an agent to an environment - note `agent=` takes the **agent ID string** and `environment_id=` is required. The **session** is the runtime that holds conversation state.

In [4]:
# 1. Create the curator agent. The system prompt sets the platformer persona.
agent = client.beta.agents.create(
    model={"id": MODEL},
    name="platformer-curator",
    system=(
        "You are a retro platformer video-game curator. "
        "When asked for a pairing, name two platformer games and give one sharp sentence on why they rhyme "
        "(level design, jump feel, music, or era). Keep it tight."
    ),
    betas=[BETA],
)

# 2. Create the environment the runtime executes in.
env = client.beta.environments.create(name="curator-scratch", betas=[BETA])

# 3. Create the session that binds this agent to this environment.
session = client.beta.sessions.create(
    agent=agent.id,               # the agent ID string, not the object
    environment_id=env.id,
    betas=[BETA],
)

print("agent:  ", agent.id)
print("env:    ", env.id)
print("session:", session.id)

agent:   agent_01FcTVynco78tKMHc2z6pEFo
env:     env_019q5KWCxe9PK7h6pjBMbDAm
session: sesn_01QH7pQW1JfzdSErjc5DsmWM


### 5. Send the first turn and run the loop until idle

We **`send`** a `user.message` event, then **`stream`** events back. The agentic loop is done when the stream yields **`session.status_idle`** - that event carries **`stop_reason`**. We collect `agent.message` text as it arrives and break on idle.

> **Gotcha.** Do not stop the loop when the assistant text "looks finished." Stop on **`session.status_idle`**. Mid-turn the stream can emit `agent.tool_use` events that the runtime still has to resolve before the turn ends; breaking early cuts the turn off before `stop_reason` is set.

In [5]:
def run_turn(text: str) -> str:
    """Send one user turn, drive the loop to idle, and return the agent's reply.

    The teaching point: the loop terminates on `session.status_idle`, whose
    `stop_reason` is the authoritative end signal - not the presence of text.
    """
    client.beta.sessions.events.send(
        session.id,
        events=[{"type": "user.message",
                 "content": [{"type": "text", "text": text}]}],
        betas=[BETA],
    )

    reply_parts: list[str] = []
    for event in client.beta.sessions.events.stream(session.id, betas=[BETA]):
        if event.type == "agent.message":
            # Accumulate assistant text as the runtime streams it.
            for block in event.content:
                if getattr(block, "type", None) == "text":
                    reply_parts.append(block.text)
        elif event.type == "session.status_idle":
            # The ONLY correct place to stop. stop_reason lives on this event.
            print("stop_reason:", event.stop_reason)
            break

    return "".join(reply_parts)


first_reply = run_turn("Pair two 16-bit-era platformers for a co-op night.")
print(first_reply)

stop_reason: BetaManagedAgentsSessionEndTurn(type='end_turn')
**Donkey Kong Country 2** and **Kirby Super Star**

Both SNES co-op classics with opposite design philosophies—one demands pixel-perfect barrel-cannon precision while the other celebrates chaotic copy-power improvisation, so you'll switch between sweating and laughing all night.


### 6. Second turn on the same session (server-side memory)

We ask a follow-up that only makes sense if the agent still remembers the first pairing. We do **not** resend the earlier games. The **session holds conversation state server-side**, so the second `send` on the same `session.id` continues the thread. This is the Domain 1 point: with Managed Agents you carry a **session ID**, not a growing local transcript.

In [6]:
# No games named here. If the agent answers correctly, the SERVER remembered
# the first turn - proof the session carries memory across turns.
second_reply = run_turn("Which of those two should we play FIRST, and why?")
print(second_reply)

stop_reason: BetaManagedAgentsSessionEndTurn(type='end_turn')
**Kirby Super Star** first.

Start loose and laughing with the chaotic copy-power mayhem, then graduate to DKC2's precision demands once you've built co-op chemistry and your fingers are warm.


### 7. Tear down (guarded)

Always **archive** what you created so no billable session dangles. There is **no `agents.delete`**; the verb is **`archive`**. Archive the session first, then the agent.

**This cell is guarded by `RUN_TEARDOWN`** (set in the client cell above). It defaults to **OFF** so a "Run All" during class leaves the agent and session **live** - handy for poking at them in the Claude Console after the demo. To actually archive, either flip the switch and re-run this one cell, or set the environment variable `ARCHIVE_ON_RUN=1` before launching (which is how the headless smoke tests force cleanup).

In [7]:
if RUN_TEARDOWN:
    client.beta.sessions.archive(session.id, betas=[BETA])
    client.beta.agents.archive(agent.id, betas=[BETA])
    print("Archived session and agent. Clean exit.")
else:
    print("Teardown SKIPPED - agent and session are still LIVE.")
    print("Archive manually: flip RUN_TEARDOWN or set ARCHIVE_ON_RUN=1 and re-run.")
    print("  agent:  ", agent.id)
    print("  session:", session.id)

Teardown SKIPPED - agent and session are still LIVE.
Archive manually: flip RUN_TEARDOWN or set ARCHIVE_ON_RUN=1 and re-run.
  agent:   agent_01FcTVynco78tKMHc2z6pEFo
  session: sesn_01QH7pQW1JfzdSErjc5DsmWM
